# TTRL + DC/ACE Co-Evolution on AIME 2024
Experiment: Does interleaved co-evolution of model weights (TTRL/GRPO) and strategy memory (DC/ACE playbook) produce synergistic improvement?

**4 Conditions:**
1. Baseline — frozen model, single generation, ground-truth check
2. DC-only — frozen model, playbook evolution, majority vote
3. TTRL-only — GRPO weight updates, majority vote, no playbook
4. DC+TTRL — co-evolution of both weights and playbook

In [ ]:
# Install dependencies (Colab)
# !pip install trl>=0.15.0 vllm>=0.7.0 transformers>=4.48 peft>=0.14 accelerate datasets torch scipy matplotlib nest_asyncio

In [ ]:
import copy
import csv
import json
import os
import re
import time
from abc import ABC, abstractmethod
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import torch
import numpy as np
import matplotlib.pyplot as plt


@dataclass
class Config:
    """All hyperparameters for the TTRL+DC experiment."""
    # Model
    MODEL_NAME: str = "Qwen/Qwen2.5-Math-7B-Instruct"

    # Generation
    NUM_GENERATIONS: int = 16
    MAX_BULLETS: int = 20

    # GRPO / RL
    KL_COEFF: float = 0.0
    LORA_RANK: int = 16
    LORA_ALPHA: int = 32
    LORA_MODULES: List[str] = field(
        default_factory=lambda: ["q_proj", "v_proj", "k_proj", "o_proj"]
    )
    NUM_EPISODES: int = 30
    LR: float = 5e-7
    MAX_GRAD_NORM: float = 1.0

    # Paths
    RESULTS_DIR: str = "results"
    CHECKPOINTS_DIR: str = "checkpoints"


CFG = Config()
os.makedirs(CFG.RESULTS_DIR, exist_ok=True)
os.makedirs(CFG.CHECKPOINTS_DIR, exist_ok=True)

print("Config:")
for k, v in vars(CFG).items():
    print(f"  {k}: {v}")

In [ ]:
# ---------------------------------------------------------------------------
# Answer parsing (reused from experiments/dc_vs_verification/run.py)
# ---------------------------------------------------------------------------

ANSWER_REGEX = re.compile(r"-?\d+(?:,\d{3})*(?:\.\d+)?")


def extract_numeric_answer(text: str) -> str:
    matches = ANSWER_REGEX.findall(text.replace(",", ""))
    if not matches:
        return text.strip()
    result = matches[-1].lstrip("0")
    return result if result else "0"


def last_boxed_only_string(string: str) -> str:
    idx = string.rfind("\\boxed")
    if idx < 0:
        idx = string.rfind("\\fbox")
    if idx < 0:
        return ""
    brace_idx = string.find("{", idx)
    if brace_idx < 0:
        return ""
    level = 0
    for i in range(brace_idx, len(string)):
        if string[i] == "{":
            level += 1
        elif string[i] == "}":
            level -= 1
            if level == 0:
                return string[idx : i + 1]
    return ""


def clean_answer(s):
    s = s.replace("\\dfrac", "\\frac")
    s = s.replace("x \\in", "")
    s = re.sub(r"\\mathbf\s*{([^}]*)}", r"\1", s)
    s = re.sub(r"\\textbf\s*{([^}]*)}", r"\1", s)
    return s


def remove_boxed(s):
    if "\\boxed " in s:
        left = "\\boxed "
        assert s[: len(left)] == left
        return s[len(left) :]
    left = "\\boxed{"
    if not s.startswith(left):
        return None
    assert s[-1] == "}"
    return clean_answer(s[len(left) : -1])


def fix_fracs(string):
    substrs = string.split("\\frac")
    new_str = substrs[0]
    if len(substrs) > 1:
        for substr in substrs[1:]:
            new_str += "\\frac"
            if substr[0] == "{":
                new_str += substr
            else:
                try:
                    assert len(substr) >= 2
                except AssertionError:
                    return string
                a, b = substr[0], substr[1]
                if b != "{":
                    new_str += "{" + a + "}{" + b + "}" + substr[2:]
                else:
                    new_str += "{" + a + "}" + b + substr[2:]
    return new_str


def fix_a_slash_b(string):
    if len(string.split("/")) != 2:
        return string
    a, b = string.split("/")
    try:
        a, b = int(a), int(b)
        assert string == "{}/{}".format(a, b)
        return "\\frac{" + str(a) + "}{" + str(b) + "}"
    except (AssertionError, ValueError):
        return string


def fix_sqrt(string):
    if "\\sqrt" not in string:
        return string
    splits = string.split("\\sqrt")
    new_string = splits[0]
    for split in splits[1:]:
        if split[0] != "{":
            new_string += "\\sqrt{" + split[0] + "}" + split[1:]
        else:
            new_string += "\\sqrt" + split
    return new_string


def remove_right_units(string):
    if "\\text{ " in string:
        splits = string.split("\\text{ ")
        assert len(splits) == 2
        return splits[0]
    return string


def strip_string(string):
    string = string.replace("\n", "")
    string = string.replace("\\!", "")
    string = string.replace("\\\\", "\\")
    string = string.replace("tfrac", "frac")
    string = string.replace("dfrac", "frac")
    string = string.replace("\\left", "")
    string = string.replace("\\right", "")
    string = string.replace("^{\\circ}", "")
    string = string.replace("^\\circ", "")
    string = string.replace("\\$", "")
    string = remove_right_units(string)
    string = string.replace("\\%", "")
    string = string.replace("%", "")
    string = string.replace(" .", " 0.")
    string = string.replace("{.", "{0.")
    if len(string) == 0:
        return string
    if string[0] == ".":
        string = "0" + string
    if len(string.split("=")) == 2:
        if len(string.split("=")[0]) <= 2:
            string = string.split("=")[1]
    string = fix_sqrt(string)
    string = string.replace(" ", "")
    string = fix_fracs(string)
    if string == "0.5":
        string = "\\frac{1}{2}"
    if string == "5.5":
        string = "\\frac{11}{2}"
    string = fix_a_slash_b(string)
    return string


def is_equiv(str1, str2, verbose=False):
    if str1 is None and str2 is None:
        return True
    if str1 is None or str2 is None:
        return False
    try:
        ss1 = strip_string(str1)
        ss2 = strip_string(str2)
        if verbose:
            print(ss1, ss2)
        return ss1 == ss2
    except Exception:
        return str1 == str2


def parse_answer(raw: str) -> str:
    """Extract answer from model output. Try \\boxed first, then #### pattern, then last number."""
    boxed = last_boxed_only_string(raw)
    if boxed:
        inner = remove_boxed(boxed)
        if inner is not None:
            return inner.strip()
    m = re.search(r"####\s*(-?[\d,]+\.?\d*)", raw)
    if m:
        return m.group(1).replace(",", "").strip()
    return extract_numeric_answer(raw)


def check_answer(predicted: str, ground_truth: str) -> bool:
    """Check if predicted answer matches ground truth."""
    if is_equiv(predicted, ground_truth):
        return True
    # AIME answers are integers 000-999; try numeric comparison
    try:
        p = int(float(predicted.replace(",", "")))
        g = int(float(ground_truth.replace(",", "")))
        return p == g
    except (ValueError, TypeError):
        return False

In [ ]:
# ---------------------------------------------------------------------------
# Component ABCs
# ---------------------------------------------------------------------------


class Generator(ABC):
    """Generates candidate solutions for a problem."""

    @abstractmethod
    def generate(self, problem: str, n: int, playbook_context: str = "") -> List[Dict]:
        """Returns list of {answer: str, raw: str, bullets_used: list}"""
        ...


class Evaluator(ABC):
    """Evaluates/selects among candidate answers."""

    @abstractmethod
    def evaluate(
        self, candidates: List[Dict], ground_truth: Optional[str] = None
    ) -> Dict:
        """Returns {selected_answer: str, reward_scores: list, metadata: dict}"""
        ...


class Curator(ABC):
    """Evolves the playbook based on results."""

    @abstractmethod
    def curate(
        self,
        playbook: Any,
        problem: str,
        solution: str,
        is_correct: bool,
        reflection: str,
    ) -> Any:
        """Returns updated playbook."""
        ...


class Trainer(ABC):
    """Updates model weights via RL."""

    @abstractmethod
    def train_step(
        self,
        prompts: List[str],
        completions: List[List[str]],
        rewards: List[List[float]],
    ) -> Dict:
        """Returns {loss: float, metrics: dict}"""
        ...


class PlaybookManager(ABC):
    """Manages playbook state and context injection."""

    @abstractmethod
    def get_context(self) -> str:
        """Returns playbook text for system prompt injection."""
        ...

    @abstractmethod
    def snapshot(self) -> Dict:
        """Returns serializable playbook state."""
        ...


class CurriculumSelector(ABC):
    """Selects/orders problems for training."""

    @abstractmethod
    def select(self, problems: List[Dict], episode: int) -> List[Dict]:
        """Returns ordered subset of problems for this episode."""
        ...


print("Component ABCs defined: Generator, Evaluator, Curator, Trainer, PlaybookManager, CurriculumSelector")

In [ ]:
# ---------------------------------------------------------------------------
# Data Loading: AIME 2024
# ---------------------------------------------------------------------------

AIME_CSV_URL = "https://raw.githubusercontent.com/ShengranHu/ADAS/main/examples/adas_aime/AIME_Dataset_1983_2025.csv"
AIME_CSV_PATH = "AIME_Dataset_1983_2025.csv"


def download_aime_data():
    """Download AIME dataset if not present."""
    if not os.path.exists(AIME_CSV_PATH):
        import urllib.request

        print(f"Downloading AIME dataset...")
        urllib.request.urlretrieve(AIME_CSV_URL, AIME_CSV_PATH)
        print(f"Downloaded to {AIME_CSV_PATH}")


def load_aime_2024() -> List[Dict]:
    """Load AIME 2024 problems from CSV."""
    download_aime_data()
    problems = []
    with open(AIME_CSV_PATH, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if str(row["Year"]).strip() == "2024":
                problems.append(
                    {
                        "id": row["ID"],
                        "problem": row["problem"],
                        "answer": str(int(row["answer"])),
                    }
                )
    return problems


# Load and verify
problems = load_aime_2024()
print(f"Loaded {len(problems)} AIME 2024 problems")
assert len(problems) == 30, f"Expected 30 problems, got {len(problems)}"

# Quick answer parsing tests
assert parse_answer("The answer is \\boxed{42}") == "42"
assert parse_answer("#### 7") == "7"
assert parse_answer("The answer is 100.") == "100"
assert check_answer("42", "42") == True
assert check_answer("042", "42") == True
print("Answer parsing tests passed!")
print(f"Sample problem: {problems[0]['id']} (answer: {problems[0]['answer']})")

In [ ]:
# ---------------------------------------------------------------------------
# Condition Configs
# ---------------------------------------------------------------------------
# Each maps component names to implementations.
# These will be filled in as we implement each component.

CONDITIONS = {
    "baseline": {
        "name": "Baseline (CoT Pass@1)",
        "playbook": "null",  # NullPlaybook
        "trainer": "none",  # No training
        "evaluator": "ground_truth",  # Direct ground-truth check
        "n_generations": 1,
        "temperature": 0.0,
    },
    "dc_only": {
        "name": "DC-only",
        "playbook": "active",  # ActivePlaybook with reflect+curate
        "trainer": "none",  # No training
        "evaluator": "majority_vote",
        "n_generations": 16,
        "temperature": 0.7,
    },
    "ttrl_only": {
        "name": "TTRL-only",
        "playbook": "null",  # NullPlaybook
        "trainer": "grpo",  # GRPO training
        "evaluator": "majority_vote",
        "n_generations": 16,
        "temperature": 0.7,
    },
    "dc_ttrl": {
        "name": "DC+TTRL",
        "playbook": "active",  # ActivePlaybook
        "trainer": "grpo",  # GRPO training
        "evaluator": "majority_vote",
        "n_generations": 16,
        "temperature": 0.7,
    },
}

print("Condition configs defined:")
for k, v in CONDITIONS.items():
    print(
        f"  {v['name']}: playbook={v['playbook']}, trainer={v['trainer']}, eval={v['evaluator']}"
    )

In [ ]:
# ---------------------------------------------------------------------------
# Playbook, Reflect/Curate Pipeline, Evaluators, Curriculum
# ---------------------------------------------------------------------------


@dataclass
class Bullet:
    id: str
    section: str
    content: str
    helpful: int = 0
    harmful: int = 0

    def to_str(self) -> str:
        return f"[{self.id}] helpful={self.helpful} harmful={self.harmful} :: {self.content}"


class Playbook:
    """Internal playbook state with bullet management."""

    def __init__(self):
        self.bullets: List[Bullet] = []
        self._next_id: int = 1

    def add(self, section: str, content: str) -> str:
        prefix = {
            "STRATEGIES": "str",
            "COMMON_MISTAKES": "err",
            "SOLUTION_PATTERNS": "sol",
        }.get(section, "gen")
        bid = f"{prefix}-{self._next_id:05d}"
        self._next_id += 1
        self.bullets.append(Bullet(id=bid, section=section, content=content))
        return bid

    def remove(self, bid: str):
        self.bullets = [b for b in self.bullets if b.id != bid]

    def update(self, bid: str, content: str):
        for b in self.bullets:
            if b.id == bid:
                b.content = content
                return

    def tag(self, bid: str, label: str):
        for b in self.bullets:
            if b.id == bid:
                if label == "helpful":
                    b.helpful += 1
                elif label == "harmful":
                    b.harmful += 1

    def to_str(self) -> str:
        sections = defaultdict(list)
        for b in self.bullets:
            sections[b.section].append(b.to_str())
        parts = []
        for sec in ["STRATEGIES", "COMMON_MISTAKES", "SOLUTION_PATTERNS"]:
            if sections[sec]:
                parts.append(f"## {sec}")
                parts.extend(sections[sec])
        return "\n".join(parts) if parts else "(empty playbook)"

    def copy(self) -> "Playbook":
        return copy.deepcopy(self)

    @property
    def size(self) -> int:
        return len(self.bullets)

    def snapshot(self) -> Dict:
        return {
            "bullets": [
                {
                    "id": b.id,
                    "section": b.section,
                    "content": b.content,
                    "helpful": b.helpful,
                    "harmful": b.harmful,
                }
                for b in self.bullets
            ],
            "next_id": self._next_id,
        }

    @classmethod
    def from_snapshot(cls, data: Dict) -> "Playbook":
        pb = cls()
        pb._next_id = data.get("next_id", 1)
        for bd in data.get("bullets", []):
            pb.bullets.append(Bullet(**bd))
        return pb


def make_initial_playbook() -> Playbook:
    pb = Playbook()
    pb.add(
        "STRATEGIES",
        "AIME problems have integer answers from 000 to 999. Always give a non-negative integer.",
    )
    pb.add(
        "STRATEGIES",
        "Break complex problems into smaller sub-problems and solve each step carefully.",
    )
    pb.add(
        "COMMON_MISTAKES",
        "Watch for off-by-one errors in counting and combinatorics problems.",
    )
    return pb


# ---------------------------------------------------------------------------
# PlaybookManager implementations
# ---------------------------------------------------------------------------


class NullPlaybook(PlaybookManager):
    """No-op playbook for baseline and TTRL-only conditions."""

    def get_context(self) -> str:
        return ""

    def snapshot(self) -> Dict:
        return {"type": "null"}

    def reflect_and_curate(self, problem, solution, is_correct, candidates, generate_fn):
        """No-op: does nothing."""
        pass


class ActivePlaybook(PlaybookManager):
    """Evolving playbook with reflect+curate pipeline."""

    def __init__(self, generate_fn):
        """
        Args:
            generate_fn: callable(system, user, temperature, max_tokens) -> str
                         Used for reflect and curate LLM calls.
        """
        self.playbook = make_initial_playbook()
        self._generate_fn = generate_fn

    def get_context(self) -> str:
        if self.playbook.size == 0:
            return ""
        return f"\nPLAYBOOK (use these strategies, reference IDs like [str-00001]):\n{self.playbook.to_str()}"

    def snapshot(self) -> Dict:
        return {"type": "active", "playbook": self.playbook.snapshot()}

    @classmethod
    def from_snapshot(cls, data: Dict, generate_fn) -> "ActivePlaybook":
        ap = cls(generate_fn)
        ap.playbook = Playbook.from_snapshot(data["playbook"])
        return ap

    def reflect_and_curate(
        self,
        problem: str,
        solution: str,
        is_correct: bool,
        candidates: List[Dict],
        generate_fn=None,
    ):
        """Run reflect then curate pipeline on best candidate."""
        fn = generate_fn or self._generate_fn

        # Find best candidate (one matching majority vote if available)
        best = candidates[0] if candidates else {"raw": solution, "bullets_used": []}
        bullets_used = best.get("bullets_used", [])
        raw_response = best.get("raw", solution)

        # --- REFLECT ---
        feedback = "CORRECT" if is_correct else "INCORRECT"
        bullets_text = "\n".join(
            f"  {b.to_str()}"
            for b in self.playbook.bullets
            if b.id in bullets_used
        )
        if not bullets_text:
            bullets_text = "  (none referenced)"

        reflect_system = (
            "You are a math reasoning analyst. Analyze the solution and whether playbook strategies helped.\n"
            'For each bullet ID used, output a JSON line: {"id": "str-00001", "tag": "helpful"}\n'
            "Tags: helpful, harmful, neutral.\n"
            "End with a reflection paragraph about what mathematical insight was key."
        )
        reflect_user = (
            f"Problem: {problem}\n\n"
            f"Solution:\n{raw_response[:2000]}\n\n"
            f"Result: {feedback}\n\n"
            f"Bullets referenced:\n{bullets_text}"
        )
        reflection = fn(reflect_system, reflect_user, temperature=0.3, max_tokens=1024)

        # Parse tags
        tags = {}
        for m in re.finditer(
            r'"id"\s*:\s*"([^"]+)".*?"tag"\s*:\s*"(helpful|harmful|neutral)"',
            reflection,
        ):
            bid, tag = m.group(1), m.group(2)
            if bid in bullets_used:
                tags[bid] = tag
        if not tags and bullets_used:
            default_tag = "helpful" if is_correct else "harmful"
            for bid in bullets_used:
                tags[bid] = default_tag
        for bid, tag in tags.items():
            self.playbook.tag(bid, tag)

        # --- CURATE ---
        pb_text = self.playbook.to_str()
        curate_system = (
            "You are a playbook curator for math competition solving. Based on the reflection, "
            "propose operations to improve the playbook.\n"
            "Output a JSON array of operations:\n"
            '[{"op": "ADD", "section": "STRATEGIES", "content": "new insight"},\n'
            ' {"op": "UPDATE", "id": "str-00001", "content": "refined text"},\n'
            ' {"op": "DELETE", "id": "err-00002"}]\n'
            f"Sections: STRATEGIES, COMMON_MISTAKES, SOLUTION_PATTERNS\n"
            f"Max bullets: {CFG.MAX_BULLETS}. Current: {self.playbook.size}.\n"
            "Only propose operations clearly supported by the reflection. Keep it minimal."
        )
        curate_user = (
            f"Question: {problem}\n"
            f"Current playbook:\n{pb_text}\n\n"
            f"Reflection:\n{reflection}"
        )
        curate_raw = fn(
            curate_system, curate_user, temperature=0.4, max_tokens=1024
        )

        # Parse JSON operations
        json_match = re.search(r"\[.*\]", curate_raw, re.DOTALL)
        if json_match:
            try:
                ops = json.loads(json_match.group())
            except json.JSONDecodeError:
                ops = []
        else:
            ops = []

        for op in ops:
            try:
                if op.get("op") == "ADD" and self.playbook.size < CFG.MAX_BULLETS:
                    self.playbook.add(
                        op.get("section", "STRATEGIES"), op.get("content", "")
                    )
                elif op.get("op") == "UPDATE" and op.get("id"):
                    self.playbook.update(op["id"], op.get("content", ""))
                elif op.get("op") == "DELETE" and op.get("id"):
                    self.playbook.remove(op["id"])
            except Exception:
                pass

        # Safety: if curate emptied the playbook, restore initial
        if self.playbook.size == 0:
            old_next = self.playbook._next_id
            self.playbook = make_initial_playbook()
            self.playbook._next_id = old_next


# ---------------------------------------------------------------------------
# Majority vote helper
# ---------------------------------------------------------------------------


def majority_vote(answers: List[str]) -> Tuple[str, float]:
    """Return (winner, confidence) where confidence = fraction of votes."""
    counter = Counter()
    for a in answers:
        try:
            normalized = str(int(float(a.replace(",", ""))))
        except (ValueError, TypeError):
            normalized = a.strip()
        counter[normalized] += 1
    if not counter:
        return "", 0.0
    winner, count = counter.most_common(1)[0]
    return winner, count / len(answers)


# ---------------------------------------------------------------------------
# Evaluator implementations
# ---------------------------------------------------------------------------


class MajorityVoteEvaluator(Evaluator):
    """Selects answer by majority vote. Assigns binary reward per candidate."""

    def evaluate(
        self, candidates: List[Dict], ground_truth: Optional[str] = None
    ) -> Dict:
        answers = [c["answer"] for c in candidates]
        winner, confidence = majority_vote(answers)
        # Reward: 1.0 if candidate matches majority, 0.0 otherwise
        rewards = []
        for c in candidates:
            try:
                norm = str(int(float(c["answer"].replace(",", ""))))
            except (ValueError, TypeError):
                norm = c["answer"].strip()
            rewards.append(1.0 if norm == winner else 0.0)
        return {
            "selected_answer": winner,
            "confidence": confidence,
            "reward_scores": rewards,
            "metadata": {"vote_distribution": dict(Counter(answers).most_common())},
        }


class GroundTruthEvaluator(Evaluator):
    """Direct ground-truth check. Used for baseline only."""

    def evaluate(
        self, candidates: List[Dict], ground_truth: Optional[str] = None
    ) -> Dict:
        if not candidates:
            return {"selected_answer": "", "reward_scores": [], "metadata": {}}
        answer = candidates[0]["answer"]
        correct = check_answer(answer, ground_truth) if ground_truth else False
        return {
            "selected_answer": answer,
            "reward_scores": [1.0 if correct else 0.0],
            "metadata": {"correct": correct},
        }


# ---------------------------------------------------------------------------
# CurriculumSelector implementation
# ---------------------------------------------------------------------------


class IdentityCurriculum(CurriculumSelector):
    """Returns all problems in original order (no curriculum)."""

    def select(self, problems: List[Dict], episode: int) -> List[Dict]:
        return problems


# ---------------------------------------------------------------------------
# Smoke tests
# ---------------------------------------------------------------------------

# Test playbook
pb = make_initial_playbook()
assert pb.size == 3
print(f"Initial playbook ({pb.size} bullets):")
print(pb.to_str())

# Test NullPlaybook
null_pb = NullPlaybook()
assert null_pb.get_context() == ""
print("\nNullPlaybook: OK")

# Test snapshot round-trip
snap = pb.snapshot()
pb2 = Playbook.from_snapshot(snap)
assert pb2.size == pb.size
assert pb2.to_str() == pb.to_str()
print("Playbook snapshot round-trip: OK")

print("\nPlaybook components ready!")

In [ ]:
# ---------------------------------------------------------------------------
# Model Loading + GRPO Training Setup
# ---------------------------------------------------------------------------
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(CFG.MODEL_NAME, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# LoRA configuration
peft_config = LoraConfig(
    r=CFG.LORA_RANK,
    lora_alpha=CFG.LORA_ALPHA,
    target_modules=CFG.LORA_MODULES,
    task_type="CAUSAL_LM",
    bias="none",
)

# System prompt builder
def build_system_prompt(playbook_context: str = "") -> str:
    base = (
        "You are an expert math competition solver. Solve the problem step-by-step.\n"
        "Show all your work clearly. At the end, put your final integer answer inside \\boxed{}.\n"
        "AIME answers are always integers from 0 to 999.\n"
    )
    if playbook_context:
        base += playbook_context
    return base

# Format prompts for the model using chat template
def format_prompt(problem: str, playbook_context: str = "") -> str:
    """Format a single problem as a chat-template prompt string."""
    system = build_system_prompt(playbook_context)
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": f"Solve this AIME problem:\n\n{problem}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# ---------------------------------------------------------------------------
# Synchronous generation for playbook reflect/curate
# ---------------------------------------------------------------------------
# This wraps whatever model is currently loaded for non-vLLM generation calls.
# The actual model reference is set before each condition runs.
_sync_model_ref = {"model": None}

def sync_generate(system: str, user: str, temperature: float = 0.3, max_tokens: int = 1024) -> str:
    """Synchronous LLM call for reflect/curate. Uses the currently loaded model."""
    model = _sync_model_ref["model"]
    if model is None:
        # Fallback: return a minimal reflection so curate can still work
        return '{"id": "none", "tag": "neutral"}\nNo model loaded for reflection.'
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True)
    inputs = inputs.to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_tokens,
            temperature=max(temperature, 0.01),  # avoid 0.0
            do_sample=temperature > 0,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return response.strip()

# ---------------------------------------------------------------------------
# GRPO reward function: majority-voting based (TTRL-style)
# ---------------------------------------------------------------------------
# GRPOTrainer calls reward_funcs with keyword args: prompts, completions,
# completion_ids, trainer_state, plus any dataset columns.
# completions is a list of generated text strings (one per generation).
# We compute binary majority-vote reward per completion.

def ttrl_reward_fn(completions, **kwargs) -> list[float]:
    """Compute majority-vote reward for a batch of completions.

    Called by GRPOTrainer. completions is a list of generated text strings
    (all generations for a single prompt in one GRPO step).
    Returns list of float rewards.
    """
    # Parse answers from all completions
    answers = [parse_answer(c) for c in completions]

    # Majority vote across the group
    winner, confidence = majority_vote(answers)

    # Binary reward: match majority = 1.0, else 0.0
    rewards = []
    for a in answers:
        try:
            norm = str(int(float(a.replace(",", ""))))
        except (ValueError, TypeError):
            norm = a.strip()
        rewards.append(1.0 if norm == winner else 0.0)

    return rewards

# ---------------------------------------------------------------------------
# DC+TTRL reward: majority-vote + playbook reflect/curate side-effect
# ---------------------------------------------------------------------------
# For the DC+TTRL condition, we need playbook operations interleaved with
# GRPO training. Since GRPOTrainer.train() handles the full loop internally,
# we embed reflect/curate as a side-effect in a custom reward function.
# The reward itself is still majority-vote binary; the side-effect updates
# the shared playbook state.

_dc_ttrl_state = {
    "playbook_mgr": None,      # ActivePlaybook instance, set before training
    "problem_lookup": {},       # prompt_text -> problem dict
    "episode_stats": [],        # track per-step stats
}

def dc_ttrl_reward_fn(prompts, completions, **kwargs) -> list[float]:
    """Majority-vote reward with playbook reflect/curate side-effect.

    This function is used for the DC+TTRL condition. In addition to computing
    the standard TTRL majority-vote reward, it runs the reflect+curate pipeline
    on the best candidate to evolve the playbook.
    """
    # Standard majority-vote reward computation
    answers = [parse_answer(c) for c in completions]
    winner, confidence = majority_vote(answers)

    rewards = []
    for a in answers:
        try:
            norm = str(int(float(a.replace(",", ""))))
        except (ValueError, TypeError):
            norm = a.strip()
        rewards.append(1.0 if norm == winner else 0.0)

    # --- Side-effect: reflect + curate on the playbook ---
    pb_mgr = _dc_ttrl_state["playbook_mgr"]
    if pb_mgr is not None and prompts:
        # Find the original problem text (best effort via lookup)
        # prompts[0] is the formatted prompt string for this group
        prompt_key = prompts[0] if isinstance(prompts, list) else str(prompts)
        problem_dict = _dc_ttrl_state["problem_lookup"].get(prompt_key, None)
        ground_truth = problem_dict["answer"] if problem_dict else None

        # Build candidates list
        candidates = []
        for c_text, a in zip(completions, answers):
            bullets_used = re.findall(r"\[(str|err|sol|gen)-\d{5}\]", c_text)
            candidates.append({"answer": a, "raw": c_text, "bullets_used": list(set(bullets_used))})

        # Check if majority winner is correct (for reflect feedback)
        is_correct = check_answer(winner, ground_truth) if ground_truth else False

        try:
            pb_mgr.reflect_and_curate(
                problem=problem_dict["problem"] if problem_dict else "(unknown)",
                solution=winner,
                is_correct=is_correct,
                candidates=candidates,
            )
        except Exception as e:
            # Don't let reflect/curate errors break training
            print(f"  [DC+TTRL] reflect/curate error: {e}")

        # Track stats
        _dc_ttrl_state["episode_stats"].append({
            "confidence": confidence,
            "is_correct": is_correct,
            "pb_size": pb_mgr.playbook.size if hasattr(pb_mgr, "playbook") else 0,
        })

    return rewards

# ---------------------------------------------------------------------------
# GRPO Training config
# ---------------------------------------------------------------------------
grpo_config = GRPOConfig(
    output_dir=CFG.CHECKPOINTS_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,       # 1 prompt at a time
    gradient_accumulation_steps=4,       # accumulate over 4 prompts
    learning_rate=CFG.LR,
    max_grad_norm=CFG.MAX_GRAD_NORM,
    bf16=True,
    logging_steps=1,
    save_strategy="no",                  # we save manually per condition
    # GRPO specific
    num_generations=CFG.NUM_GENERATIONS, # 16 generations per prompt
    max_completion_length=2048,
    max_prompt_length=2048,
    # vLLM colocate mode (shared GPU inference + training)
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.5,     # 50% vLLM, 50% training
    # RL params
    beta=CFG.KL_COEFF,                   # 0.0 per TTRL paper
    report_to="none",
)

# NullTrainer for non-training conditions
class NullTrainer(Trainer):
    """No-op trainer for baseline and DC-only conditions."""
    def train_step(self, prompts, completions, rewards) -> Dict:
        return {"loss": 0.0, "metrics": {}}

print("GRPO config ready:")
print(f"  num_generations:             {grpo_config.num_generations}")
print(f"  learning_rate:               {grpo_config.learning_rate}")
print(f"  beta (KL):                   {grpo_config.beta}")
print(f"  bf16:                        {grpo_config.bf16}")
print(f"  vllm_mode:                   {grpo_config.vllm_mode}")
print(f"  vllm_gpu_memory_utilization: {grpo_config.vllm_gpu_memory_utilization}")
print(f"  LoRA rank: {peft_config.r}, alpha: {peft_config.lora_alpha}")
print(f"  Target modules: {peft_config.target_modules}")
print(f"Tokenizer: pad_token={tokenizer.pad_token!r}")

In [ ]:
# ---------------------------------------------------------------------------
# Co-Evolution Experiment Loop
# ---------------------------------------------------------------------------
from datasets import Dataset


def run_frozen_condition(condition_key: str, problems: List[Dict],
                         model, tokenizer) -> Dict:
    """Run a non-training condition (baseline or DC-only) with a frozen model.

    Generates candidates via model.generate(), evaluates, and optionally
    evolves the playbook. No weight updates occur.

    Args:
        condition_key: 'baseline' or 'dc_only'
        problems: list of AIME problem dicts
        model: loaded HF model (frozen, on GPU)
        tokenizer: loaded tokenizer

    Returns:
        dict with per-episode results and playbook snapshots
    """
    cond = CONDITIONS[condition_key]
    print(f"\n{'='*60}")
    print(f"Running: {cond['name']}")
    print(f"{'='*60}")

    use_playbook = cond["playbook"] == "active"
    n_gen = cond["n_generations"]
    temp = cond["temperature"]

    # Set up sync_generate model reference for reflect/curate
    _sync_model_ref["model"] = model

    # Initialize playbook
    if use_playbook:
        playbook_mgr = ActivePlaybook(sync_generate)
    else:
        playbook_mgr = NullPlaybook()

    # Initialize evaluator
    if cond["evaluator"] == "majority_vote":
        evaluator = MajorityVoteEvaluator()
    else:
        evaluator = GroundTruthEvaluator()

    all_episode_results = []
    playbook_snapshots = []

    for episode in range(CFG.NUM_EPISODES):
        episode_start = time.time()
        episode_correct = 0
        episode_total = 0
        episode_details = []

        for p_idx, problem in enumerate(problems):
            playbook_context = playbook_mgr.get_context()
            prompt_text = format_prompt(problem["problem"], playbook_context)

            # Generate N candidates
            candidates = []
            with torch.no_grad():
                for _ in range(n_gen):
                    inputs = tokenizer(
                        prompt_text, return_tensors="pt",
                        padding=True, truncation=True, max_length=2048,
                    )
                    inputs = {k: v.to(model.device) for k, v in inputs.items()}
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=2048,
                        temperature=max(temp, 0.01) if n_gen > 1 else 0.01,
                        do_sample=n_gen > 1,
                        pad_token_id=tokenizer.pad_token_id,
                    )
                    text = tokenizer.decode(
                        outputs[0][inputs["input_ids"].shape[1]:],
                        skip_special_tokens=True,
                    )
                    answer = parse_answer(text)
                    bullets_used = re.findall(
                        r"\[(str|err|sol|gen)-\d{5}\]", text
                    )
                    candidates.append({
                        "answer": answer,
                        "raw": text,
                        "bullets_used": list(set(bullets_used)),
                    })

            # Evaluate
            eval_result = evaluator.evaluate(
                candidates, ground_truth=problem["answer"]
            )
            selected = eval_result["selected_answer"]
            is_correct = check_answer(selected, problem["answer"])

            # Playbook reflect+curate (DC-only condition)
            if use_playbook:
                playbook_mgr.reflect_and_curate(
                    problem=problem["problem"],
                    solution=selected,
                    is_correct=is_correct,
                    candidates=candidates,
                )

            episode_correct += int(is_correct)
            episode_total += 1

            episode_details.append({
                "problem_id": problem["id"],
                "selected_answer": selected,
                "ground_truth": problem["answer"],
                "correct": is_correct,
                "n_candidates": len(candidates),
                "confidence": eval_result.get("confidence", None),
            })

        # Save playbook snapshot
        playbook_snapshots.append(playbook_mgr.snapshot())

        episode_acc = episode_correct / episode_total if episode_total > 0 else 0.0
        episode_time = time.time() - episode_start

        all_episode_results.append({
            "episode": episode,
            "accuracy": episode_acc,
            "correct": episode_correct,
            "total": episode_total,
            "time_s": episode_time,
            "details": episode_details,
        })

        pb_bullets = playbook_mgr.snapshot().get("playbook", {}).get("bullets", [])
        pb_count = len(pb_bullets) if isinstance(pb_bullets, list) else 0
        print(f"  Episode {episode+1}/{CFG.NUM_EPISODES}: "
              f"acc={episode_acc:.1%} ({episode_correct}/{episode_total}) "
              f"pb_size={pb_count} time={episode_time:.0f}s")

    return {
        "condition": condition_key,
        "name": cond["name"],
        "episodes": all_episode_results,
        "playbook_snapshots": playbook_snapshots,
        "training_metrics": [],
    }


def evaluate_trained_model(condition_key: str, problems: List[Dict],
                           grpo_trainer, tokenizer) -> Dict:
    """Evaluate a trained model (post-GRPO) on all problems for NUM_EPISODES.

    After GRPOTrainer.train() finishes, this function runs inference using
    the trained model to measure per-episode accuracy. For TTRL-only, no
    playbook is used. The model weights are frozen during evaluation.

    Args:
        condition_key: 'ttrl_only' (or could be reused)
        problems: list of AIME problem dicts
        grpo_trainer: trained GRPOTrainer instance
        tokenizer: loaded tokenizer

    Returns:
        dict with per-episode results
    """
    cond = CONDITIONS[condition_key]
    print(f"\n{'='*60}")
    print(f"Evaluating: {cond['name']} (post-training)")
    print(f"{'='*60}")

    n_gen = cond["n_generations"]
    temp = cond["temperature"]

    evaluator = MajorityVoteEvaluator()
    trained_model = grpo_trainer.model
    trained_model.eval()

    all_episode_results = []

    for episode in range(CFG.NUM_EPISODES):
        episode_start = time.time()
        episode_correct = 0
        episode_total = 0
        episode_details = []

        for p_idx, problem in enumerate(problems):
            prompt_text = format_prompt(problem["problem"])

            # Generate N candidates with trained model
            candidates = []
            with torch.no_grad():
                for _ in range(n_gen):
                    inputs = tokenizer(
                        prompt_text, return_tensors="pt",
                        padding=True, truncation=True, max_length=2048,
                    )
                    inputs = {k: v.to(trained_model.device) for k, v in inputs.items()}
                    outputs = trained_model.generate(
                        **inputs,
                        max_new_tokens=2048,
                        temperature=max(temp, 0.01),
                        do_sample=True,
                        pad_token_id=tokenizer.pad_token_id,
                    )
                    text = tokenizer.decode(
                        outputs[0][inputs["input_ids"].shape[1]:],
                        skip_special_tokens=True,
                    )
                    answer = parse_answer(text)
                    candidates.append({
                        "answer": answer,
                        "raw": text,
                        "bullets_used": [],
                    })

            eval_result = evaluator.evaluate(
                candidates, ground_truth=problem["answer"]
            )
            selected = eval_result["selected_answer"]
            is_correct = check_answer(selected, problem["answer"])

            episode_correct += int(is_correct)
            episode_total += 1

            episode_details.append({
                "problem_id": problem["id"],
                "selected_answer": selected,
                "ground_truth": problem["answer"],
                "correct": is_correct,
                "n_candidates": len(candidates),
                "confidence": eval_result.get("confidence", None),
            })

        episode_acc = episode_correct / episode_total if episode_total > 0 else 0.0
        episode_time = time.time() - episode_start

        all_episode_results.append({
            "episode": episode,
            "accuracy": episode_acc,
            "correct": episode_correct,
            "total": episode_total,
            "time_s": episode_time,
            "details": episode_details,
        })

        print(f"  Episode {episode+1}/{CFG.NUM_EPISODES}: "
              f"acc={episode_acc:.1%} ({episode_correct}/{episode_total}) "
              f"time={episode_time:.0f}s")

    return {
        "condition": condition_key,
        "name": cond["name"],
        "episodes": all_episode_results,
        "playbook_snapshots": [{"type": "null"}] * CFG.NUM_EPISODES,
        "training_metrics": [],
    }


# ---------------------------------------------------------------------------
# Run all 4 conditions
# ---------------------------------------------------------------------------
print("=" * 60)
print("TTRL + DC/ACE Co-Evolution Experiment")
print(f"Model: {CFG.MODEL_NAME}")
print(f"Problems: {len(problems)} AIME 2024")
print(f"Episodes: {CFG.NUM_EPISODES}")
print(f"Generations/prompt: {CFG.NUM_GENERATIONS}")
print("=" * 60)

all_results = {}

# ===================================================================
# Condition 1: Baseline (frozen model, single gen, ground-truth eval)
# ===================================================================
print("\n[1/4] Loading model for Baseline + DC-only conditions...")
model = AutoModelForCausalLM.from_pretrained(
    CFG.MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"Model loaded: {CFG.MODEL_NAME}")

result_baseline = run_frozen_condition("baseline", problems, model, tokenizer)
all_results["baseline"] = result_baseline

with open(os.path.join(CFG.RESULTS_DIR, "baseline.json"), "w") as f:
    json.dump(result_baseline, f, indent=2, default=str)
print(f"Baseline saved. Final acc: {result_baseline['episodes'][-1]['accuracy']:.1%}")

# ===================================================================
# Condition 2: DC-only (frozen model, playbook evolution, maj vote)
# ===================================================================
result_dc = run_frozen_condition("dc_only", problems, model, tokenizer)
all_results["dc_only"] = result_dc

with open(os.path.join(CFG.RESULTS_DIR, "dc_only.json"), "w") as f:
    json.dump(result_dc, f, indent=2, default=str)
print(f"DC-only saved. Final acc: {result_dc['episodes'][-1]['accuracy']:.1%}")

# Free base model before GRPO training
del model
_sync_model_ref["model"] = None
torch.cuda.empty_cache()

# ===================================================================
# Condition 3: TTRL-only (GRPO weight updates, maj vote, no playbook)
# ===================================================================
print("\n[3/4] Initializing GRPOTrainer for TTRL-only...")

# Build training dataset: prompts without playbook context
train_prompts = [format_prompt(p["problem"]) for p in problems]
train_dataset = Dataset.from_dict({"prompt": train_prompts})

grpo_trainer_ttrl = GRPOTrainer(
    model=CFG.MODEL_NAME,
    reward_funcs=ttrl_reward_fn,
    args=grpo_config,
    train_dataset=train_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)
print("GRPOTrainer initialized (TTRL-only, vLLM colocate mode)")

# Run GRPO training: GRPOTrainer.train() handles the full loop including
# vLLM sleep/wake cycle for colocate mode. Each "epoch" processes all 30
# prompts with num_generations=16 candidates each.
print("Starting TTRL-only training...")
try:
    grpo_trainer_ttrl.train()
    print("TTRL-only training complete!")
except Exception as e:
    print(f"TTRL-only training error: {e}")
    import traceback; traceback.print_exc()

# Evaluate the trained model
result_ttrl = evaluate_trained_model(
    "ttrl_only", problems, grpo_trainer_ttrl, tokenizer
)
all_results["ttrl_only"] = result_ttrl

with open(os.path.join(CFG.RESULTS_DIR, "ttrl_only.json"), "w") as f:
    json.dump(result_ttrl, f, indent=2, default=str)
print(f"TTRL-only saved. Final acc: {result_ttrl['episodes'][-1]['accuracy']:.1%}")

# Save TTRL checkpoint
grpo_trainer_ttrl.save_model(os.path.join(CFG.CHECKPOINTS_DIR, "ttrl_only_final"))

# Clean up
del grpo_trainer_ttrl
torch.cuda.empty_cache()

# ===================================================================
# Condition 4: DC+TTRL (co-evolution of weights + playbook)
# ===================================================================
print("\n[4/4] Initializing GRPOTrainer for DC+TTRL...")

# Set up the DC+TTRL shared state: playbook manager + problem lookup
_dc_ttrl_state["playbook_mgr"] = ActivePlaybook(sync_generate)
_dc_ttrl_state["episode_stats"] = []

# Build problem lookup: formatted prompt -> problem dict
# This lets dc_ttrl_reward_fn find the original problem for reflect/curate
for p in problems:
    prompt_key = format_prompt(p["problem"])
    _dc_ttrl_state["problem_lookup"][prompt_key] = p

# For DC+TTRL, playbook context evolves during training. The initial prompts
# are formatted without playbook context (GRPOTrainer will use these to
# generate candidates). The reward function handles playbook operations.
# Note: as training progresses, the playbook evolves but the prompts in
# the dataset remain fixed (no playbook context in prompts). The playbook
# influence comes through the reward function side-effects that teach the
# model strategies indirectly via reward shaping.
train_dataset_dcttrl = Dataset.from_dict({"prompt": train_prompts})

grpo_trainer_dcttrl = GRPOTrainer(
    model=CFG.MODEL_NAME,
    reward_funcs=dc_ttrl_reward_fn,
    args=grpo_config,
    train_dataset=train_dataset_dcttrl,
    peft_config=peft_config,
    processing_class=tokenizer,
)
print("DC+TTRL GRPOTrainer initialized")

# Set sync_generate model ref to the DC+TTRL trainer's model for reflect/curate
_sync_model_ref["model"] = grpo_trainer_dcttrl.model

print("Starting DC+TTRL training...")
try:
    grpo_trainer_dcttrl.train()
    print("DC+TTRL training complete!")
except Exception as e:
    print(f"DC+TTRL training error: {e}")
    import traceback; traceback.print_exc()

# Evaluate the DC+TTRL trained model
result_dcttrl = evaluate_trained_model(
    "dc_ttrl", problems, grpo_trainer_dcttrl, tokenizer
)
# Attach playbook snapshots and DC stats from training
result_dcttrl["playbook_snapshots"] = [
    _dc_ttrl_state["playbook_mgr"].snapshot()
]
result_dcttrl["dc_ttrl_training_stats"] = _dc_ttrl_state["episode_stats"]
all_results["dc_ttrl"] = result_dcttrl

with open(os.path.join(CFG.RESULTS_DIR, "dc_ttrl.json"), "w") as f:
    json.dump(result_dcttrl, f, indent=2, default=str)
print(f"DC+TTRL saved. Final acc: {result_dcttrl['episodes'][-1]['accuracy']:.1%}")

# Save DC+TTRL checkpoint
grpo_trainer_dcttrl.save_model(os.path.join(CFG.CHECKPOINTS_DIR, "dc_ttrl_final"))

# Save final playbook
with open(os.path.join(CFG.RESULTS_DIR, "dc_ttrl_playbook_final.json"), "w") as f:
    json.dump(_dc_ttrl_state["playbook_mgr"].snapshot(), f, indent=2)

# Clean up
del grpo_trainer_dcttrl
_sync_model_ref["model"] = None
torch.cuda.empty_cache()

# ===================================================================
# Save combined results
# ===================================================================
with open(os.path.join(CFG.RESULTS_DIR, "all_results.json"), "w") as f:
    json.dump(all_results, f, indent=2, default=str)

print("\n" + "=" * 60)
print("All conditions complete! Results saved to", CFG.RESULTS_DIR)
print("=" * 60)
for k, v in all_results.items():
    final_acc = v["episodes"][-1]["accuracy"]
    print(f"  {v['name']:25s}: {final_acc:.1%}")

In [ ]:
# ---------------------------------------------------------------------------
# Group D: Analysis & Decision (REQ-6)
# ---------------------------------------------------------------------------
from scipy import stats

# Load results if running analysis cell independently
if "all_results" not in dir() or not all_results:
    with open(os.path.join(CFG.RESULTS_DIR, "all_results.json")) as f:
        all_results = json.load(f)

# ---------------------------------------------------------------------------
# Bootstrap CI function
# ---------------------------------------------------------------------------

def bootstrap_ci(data, n_bootstrap=10000, ci=0.95, stat_fn=np.mean):
    """Bootstrap confidence interval for a statistic."""
    rng = np.random.default_rng(42)
    boot_stats = []
    data = np.array(data)
    for _ in range(n_bootstrap):
        sample = rng.choice(data, size=len(data), replace=True)
        boot_stats.append(stat_fn(sample))
    boot_stats = np.array(boot_stats)
    lower = np.percentile(boot_stats, (1 - ci) / 2 * 100)
    upper = np.percentile(boot_stats, (1 + ci) / 2 * 100)
    return float(np.mean(boot_stats)), float(lower), float(upper)

# ---------------------------------------------------------------------------
# Extract per-problem binary outcomes for the last episode
# ---------------------------------------------------------------------------

conditions = ["baseline", "dc_only", "ttrl_only", "dc_ttrl"]
condition_labels = {
    "baseline": "Baseline",
    "dc_only": "DC-only",
    "ttrl_only": "TTRL-only",
    "dc_ttrl": "DC+TTRL",
}

last_ep = CFG.NUM_EPISODES - 1
binary_outcomes = {}
for cond in conditions:
    details = all_results[cond]["episodes"][last_ep]["details"]
    binary_outcomes[cond] = [d["correct"] for d in details]

# ---------------------------------------------------------------------------
# McNemar's test for paired comparisons
# ---------------------------------------------------------------------------

def mcnemar_test(outcomes_a, outcomes_b):
    """McNemar's test on paired binary outcomes."""
    a = np.array(outcomes_a, dtype=bool)
    b = np.array(outcomes_b, dtype=bool)
    # Contingency: a_right_b_wrong vs a_wrong_b_right
    n01 = np.sum(~a & b)  # a wrong, b right
    n10 = np.sum(a & ~b)  # a right, b wrong
    # McNemar's chi-squared (with continuity correction)
    if n01 + n10 == 0:
        return 1.0, 0, 0
    chi2 = (abs(n01 - n10) - 1) ** 2 / (n01 + n10)
    p_value = 1 - stats.chi2.cdf(chi2, df=1)
    return float(p_value), int(n01), int(n10)

# Key comparisons
print("=" * 60)
print("Statistical Analysis")
print("=" * 60)

comparisons = [
    ("dc_ttrl", "ttrl_only", "DC+TTRL vs TTRL-only"),
    ("dc_ttrl", "dc_only", "DC+TTRL vs DC-only"),
    ("dc_ttrl", "baseline", "DC+TTRL vs Baseline"),
    ("ttrl_only", "baseline", "TTRL-only vs Baseline"),
    ("dc_only", "baseline", "DC-only vs Baseline"),
]

print(f"\n{'Comparison':35s} {'p-value':>10s} {'n01':>5s} {'n10':>5s} {'Sig':>5s}")
print("-" * 65)
for cond_a, cond_b, label in comparisons:
    p_val, n01, n10 = mcnemar_test(binary_outcomes[cond_a], binary_outcomes[cond_b])
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
    print(f"{label:35s} {p_val:10.4f} {n01:5d} {n10:5d} {sig:>5s}")

# ---------------------------------------------------------------------------
# Bootstrap CIs on final accuracy
# ---------------------------------------------------------------------------

print(f"\n{'Condition':15s} {'Accuracy':>10s} {'95% CI':>20s}")
print("-" * 50)
for cond in conditions:
    outcomes = binary_outcomes[cond]
    mean, lo, hi = bootstrap_ci(outcomes, n_bootstrap=10000)
    print(f"{condition_labels[cond]:15s} {mean:10.1%} [{lo:.1%}, {hi:.1%}]")

# Key difference: DC+TTRL minus TTRL-only
dcttrl_outcomes = np.array(binary_outcomes["dc_ttrl"], dtype=float)
ttrl_outcomes = np.array(binary_outcomes["ttrl_only"], dtype=float)
diff = dcttrl_outcomes - ttrl_outcomes
diff_mean, diff_lo, diff_hi = bootstrap_ci(diff, n_bootstrap=10000)
print(f"\nDC+TTRL - TTRL-only: {diff_mean:+.1%} [{diff_lo:+.1%}, {diff_hi:+.1%}]")

# ---------------------------------------------------------------------------
# Plots: 2x2 grid
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = {"baseline": "#888888", "dc_only": "#2196F3", "ttrl_only": "#FF9800", "dc_ttrl": "#4CAF50"}

# (a) Accuracy over episodes
ax = axes[0, 0]
for cond in conditions:
    accs = [ep["accuracy"] for ep in all_results[cond]["episodes"]]
    ax.plot(range(1, len(accs) + 1), accs, label=condition_labels[cond],
            color=colors[cond], linewidth=2)
ax.set_xlabel("Episode")
ax.set_ylabel("Accuracy")
ax.set_title("(a) Accuracy Over Episodes")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

# (b) Playbook size over time (DC conditions only)
ax = axes[0, 1]
for cond in ["dc_only", "dc_ttrl"]:
    snapshots = all_results[cond].get("playbook_snapshots", [])
    sizes = []
    for snap in snapshots:
        if snap.get("type") == "null":
            sizes.append(0)
        elif "playbook" in snap:
            sizes.append(len(snap["playbook"].get("bullets", [])))
        else:
            sizes.append(0)
    if sizes:
        ax.plot(range(1, len(sizes) + 1), sizes, label=condition_labels[cond],
                color=colors[cond], linewidth=2)
ax.set_xlabel("Episode")
ax.set_ylabel("Playbook Size (bullets)")
ax.set_title("(b) Playbook Size Over Time")
ax.legend()
ax.grid(True, alpha=0.3)

# (c) Reward accuracy: how often does majority vote match ground truth?
ax = axes[1, 0]
for cond in ["dc_only", "ttrl_only", "dc_ttrl"]:
    episodes = all_results[cond]["episodes"]
    reward_accs = []
    for ep in episodes:
        # For each episode, majority vote accuracy = episode accuracy
        # (since majority vote IS the selection mechanism)
        reward_accs.append(ep["accuracy"])
    ax.plot(range(1, len(reward_accs) + 1), reward_accs, label=condition_labels[cond],
            color=colors[cond], linewidth=2)
ax.set_xlabel("Episode")
ax.set_ylabel("Majority Vote Accuracy")
ax.set_title("(c) Reward Signal Quality Over Episodes")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

# (d) Final accuracy bar chart with confidence intervals
ax = axes[1, 1]
x_pos = np.arange(len(conditions))
means = []
ci_lower = []
ci_upper = []
bar_colors = [colors[c] for c in conditions]

for cond in conditions:
    outcomes = binary_outcomes[cond]
    mean, lo, hi = bootstrap_ci(outcomes, n_bootstrap=10000)
    means.append(mean)
    ci_lower.append(mean - lo)
    ci_upper.append(hi - mean)

bars = ax.bar(x_pos, means, color=bar_colors, alpha=0.8, edgecolor="black")
ax.errorbar(x_pos, means, yerr=[ci_lower, ci_upper], fmt="none",
            ecolor="black", capsize=5, linewidth=1.5)
ax.set_xticks(x_pos)
ax.set_xticklabels([condition_labels[c] for c in conditions], rotation=15)
ax.set_ylabel("Accuracy")
ax.set_title("(d) Final Accuracy (Last Episode) with 95% CI")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis="y")

# Add value labels
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.02,
            f"{mean:.1%}", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(CFG.RESULTS_DIR, "analysis_plots.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Plots saved to {os.path.join(CFG.RESULTS_DIR, 'analysis_plots.png')}")

# ---------------------------------------------------------------------------
# Decision verdict
# ---------------------------------------------------------------------------

print("\n" + "=" * 60)
print("DECISION VERDICT")
print("=" * 60)

dcttrl_acc = all_results["dc_ttrl"]["episodes"][-1]["accuracy"]
ttrl_acc = all_results["ttrl_only"]["episodes"][-1]["accuracy"]
dc_acc = all_results["dc_only"]["episodes"][-1]["accuracy"]
base_acc = all_results["baseline"]["episodes"][-1]["accuracy"]

gap = dcttrl_acc - ttrl_acc
print(f"  Baseline:     {base_acc:.1%}")
print(f"  DC-only:      {dc_acc:.1%}")
print(f"  TTRL-only:    {ttrl_acc:.1%}")
print(f"  DC+TTRL:      {dcttrl_acc:.1%}")
print(f"  Gap (DC+TTRL - TTRL-only): {gap:+.1%}")
print()

if gap > 0.10:
    verdict = "CO-EVOLUTION WINS"
    next_step = "Co-evolution produces synergistic improvement. Scale up: more episodes, larger model, add strong-model verifier."
elif gap > 0.05:
    verdict = "MARGINAL SYNERGY"
    next_step = "Small improvement from co-evolution. Try: inject playbook into prompts during training, or use verifier-filtered reward."
elif gap > -0.05:
    verdict = "GRPO SUBSUMES PLAYBOOK"
    next_step = "Weight updates capture what the playbook provides. Focus on TTRL improvements: more data, curriculum, reward engineering."
else:
    verdict = "PLAYBOOK INTERFERES WITH RL"
    next_step = "Playbook operations hurt training. Investigate: reward noise from reflect/curate, or playbook poisoning under RL."

print(f"  VERDICT: {verdict}")
print(f"  NEXT:    {next_step}")

# Save analysis
analysis = {
    "final_accuracies": {cond: all_results[cond]["episodes"][-1]["accuracy"] for cond in conditions},
    "gap_dcttrl_minus_ttrl": gap,
    "verdict": verdict,
    "next_step": next_step,
}
with open(os.path.join(CFG.RESULTS_DIR, "analysis.json"), "w") as f:
    json.dump(analysis, f, indent=2)
print(f"\nAnalysis saved to {os.path.join(CFG.RESULTS_DIR, 'analysis.json')}")